# 00 - Explore the data

A first look at the runs: one run in long and wide format, the two queues over time, the distribution of the bottleneck depth, lost samples, and how quickly the queue forgets its past.

Only **training runs** are used here. Validation runs are for choosing models; test runs are used once, at the end.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "extras" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from extras import windows
from src import load

splits = load.load_splits()
{name: len(runs) for name, runs in splits.items()}

## One run in long format

One row per received sample and switch: `hop` 0 is the bottleneck switch, `hop` 1 the downstream switch. See `data/DATA_CARD.md` for the columns.

In [ ]:
run_id = splits["train"][0]
run = load.load_run(run_id)
run.head(6)

In [ ]:
run.groupby("hop")[["qdepth_pkts", "qdelay_us"]].describe().T

## Wide format

`load.to_wide` gives one row per sample (`seq`), with both switches side by side. A lost sample is a row of NaN, so row *i* is always sample *i*, 2 ms apart.

In [ ]:
wide = load.to_wide(run)
print(f"{len(wide)} samples, {wide[load.TARGET].isna().sum()} lost")
wide.head()

## Ten seconds of both queues

In [ ]:
part = wide.iloc[:5000]
seconds = part.index * load.SAMPLE_MS / 1000
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(seconds, part["qdepth_pkts_h0"], lw=0.6)
axes[0].axhline(load.BUFFER_PKTS, color="grey", ls="--", lw=0.8)
axes[0].set_ylabel("bottleneck depth (pkts)")
axes[1].plot(seconds, part["qdepth_pkts_h1"], lw=0.6, color="tab:orange")
axes[1].set_ylabel("downstream depth (pkts)")
axes[1].set_xlabel("time (s)")
fig.suptitle(f"{run_id}: first 10 s")
plt.tight_layout()

## Bottleneck depth over all training runs

In [ ]:
train = windows.load_wide(splits["train"])
depth = pd.concat([w[load.TARGET] for w in train.values()]).dropna()
print(f"{len(depth)} samples; empty queue: {(depth == 0).mean():.1%}; "
      f"60 packets or more: {(depth >= 60).mean():.1%}")
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(depth, bins=np.arange(load.BUFFER_PKTS + 2) - 0.5, log=True)
ax.set_xlabel("bottleneck depth (pkts)")
ax.set_ylabel("samples (log scale)");

## Lost samples

Lost samples are gaps in `seq`. They are not missing at random: most come right after a reading close to the 64-packet limit, so dropping them removes the high end of the target.

In [ ]:
rows = {}
for rid, w in train.items():
    q = w[load.TARGET]
    before = q.ffill().shift(1)[q.isna()]
    rows[rid] = {"lost": int(q.isna().sum()), "share after depth >= 60": (before >= 60).mean()}
lost = pd.DataFrame(rows).T
lost.describe()

## How fast does the queue forget?

Correlation of the bottleneck depth with itself *k* samples later, averaged over the training runs. The dashed lines are the forecast horizons (1, 5, 10, 25 samples).

In [ ]:
ks = np.arange(1, 101)
def self_corr(x, k):
    x = x.ffill().to_numpy()
    return np.corrcoef(x[:-k], x[k:])[0, 1]

corr = np.mean([[self_corr(w[load.TARGET], k) for k in ks] for w in train.values()], axis=0)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(ks, corr)
for h in windows.HORIZONS:
    ax.axvline(h, color="grey", ls="--", lw=0.8)
ax.set_xlabel("k (samples, 2 ms each)")
ax.set_ylabel("correlation");

## Supervised windows

`windows.make_windows` cuts a wide run into inputs `X` (the last `lookback` samples of each feature) and targets `Y` (bottleneck depth at t + 1, 5, 10, 25). NaN marks lost samples; how to fill them is a modelling choice. `windows.load_windows("train")` does this for a whole split and also returns the run of each row, for cross-validation grouped by run.

In [ ]:
X, Y, origin = windows.make_windows(wide, lookback=50)
print("X", X.shape, "Y", Y.shape)
pd.Series(np.isnan(Y).mean(axis=0), index=windows.HORIZONS, name="share of NaN targets")